# W02-B · Boolean search challenge *(~50 min)*

**Scenario.** The Appliances & Parts category team wants a search box over its 30,000 reviews, so that anyone can ask
*"which reviews mention the ice maker leaking?"* and get the answer in a blink. You will build the engine behind it,
the same way every search engine starts:

1. a term–document **incidence matrix**, on a toy collection, to see the idea
2. an **inverted index** over all 30,000 reviews
3. the **merge** algorithms for AND, OR and AND NOT
4. a query planner that does the least work
5. then use it to answer the category manager's questions

Every ✍️ task checks itself against a slow brute-force answer. Task 7 is a stretch.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ads6703-nlp/blob/main/w02/W02-B-boolean-search-challenge.ipynb) · On Colab, run the next cell first. On your laptop, you can skip it.

In [ ]:
# Only for Google Colab: fetch the course folder (with the data) and move into this week's folder.
# On your laptop this cell does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if os.path.exists("/content/ADS6703"):
        subprocess.run(["git", "-C", "/content/ADS6703", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/shripada/ads6703-nlp.git", "/content/ADS6703"], check=True)
    os.chdir("/content/ADS6703/w02")

In [ ]:
import re                          # the tokeniser pattern
import time                        # timing the search
from functools import lru_cache    # remember each word's stem, so we stem each type only once
import pandas as pd                # the review table
import nltk
from nltk.stem import PorterStemmer

pd.set_option("display.max_colwidth", 160)   # show 160 characters per cell, so review text isn't cut off

DATA = "../data/"   # notebooks live in ADS6703/w02/, data in ADS6703/data/
reviews = pd.read_csv(DATA + "reviews.csv.gz")
reviews["clean"] = reviews["text"].fillna("").str.replace(r"<br\s*/?>", " ", regex=True)   # drop <br /> as in W1

stemmer = PorterStemmer()

def tokenize(text):
    # W02-A, Exercise 1: lower-case, then runs of letters/digits with an optional 's or 't ending
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text.lower())

@lru_cache(maxsize=None)   # a cache: the second time we see a word, return the stored stem instead of recomputing it
def stem(word):
    return stemmer.stem(word)

def normalise(text):
    # The index terms of a text: its tokens, each Porter-stemmed. Queries go through the same function,
    # so "Leaking" in a query and "leaked" in a review both become the term "leak".
    return [stem(t) for t in tokenize(text)]

t0 = time.time()
doc_terms = [normalise(text) for text in reviews["clean"]]   # one list of terms per review; the list position is the doc ID
print(f"normalised {len(doc_terms):,} reviews in {time.time() - t0:.1f} s")
print(normalise("The ice maker LEAKED, and now it is leaking again!"))

In [ ]:
def show(doc_ids, n=3, width=150):
    # Print the first n reviews in a list of doc IDs: ID, star rating, and the start of the text
    print(f"{len(doc_ids)} results")
    for d in list(doc_ids)[:n]:
        print(f"  [{d}] {reviews.at[d, 'rating']}★  {reviews.at[d, 'clean'][:width]}")

## Task 1 · The incidence matrix (8 min)

Start with the four reviews from the slides. An **incidence matrix** has one **row per term** and one **column per
document**, with 1 (True) where the term appears in the document.

In [ ]:
toy = {
    "D1": "Filter leaks after a week.",
    "D2": "Great filter. No leaks!",
    "D3": "Ice maker leaked water.",
    "D4": "Great price, fast delivery.",
}
# For each document, a dict {term: True} for the terms it contains. pandas lines the dicts up as columns,
# fills the gaps with False, and sort_index puts the terms in alphabetical order.
M = pd.DataFrame({d: {t: True for t in normalise(text)} for d, text in toy.items()}).fillna(False).astype(bool).sort_index()
M.astype(int)   # shown as 1/0, easier to read than True/False

> 📖 **Reading the output.** 13 rows (terms) × 4 columns (documents). Normalisation already did its job: *leaks* and
> *leaked* are both in the row `leak`. Each row is a **bit vector**: `leak` is `1 1 1 0`, present in D1, D2 and D3.

A Boolean query is answered by combining rows: `&` is AND, `|` is OR, `~` is NOT, applied column by column.

### ✍️ Exercise 1: `leak AND NOT ice`

Combine the rows `M.loc["leak"]` and `M.loc["ice"]` into `answer`, a True/False Series over D1–D4.

In [ ]:
answer = ...   # ✍️ combine the two rows with & and ~
print(answer.astype(int).to_dict())
assert list(answer[answer].index) == ["D1", "D2"], "not quite: D1 and D2 mention a leak and not the ice maker"
print("✅ Now read D2 again. Is it a review about a leak?")

The matrix is simple, but how big would it be for all 30,000 reviews? We'll answer that in Task 2, once we know how
many terms there are.

## Task 2 · The inverted index (8 min)

An **inverted index** stores only the 1s: for each term, the sorted list of IDs of the documents that contain it.
That list is the term's **postings list**. The doc ID of a review is its position in `doc_terms` (0 to 29,999).

### ✍️ Exercise 2: build it

Write `build_index(doc_terms)`, returning a dict `{term: [doc IDs]}`. Each doc ID must appear **once** per term, even if the
term appears several times in that review. (W02-0, section 5, built exactly this for four reviews.)

In [ ]:
def build_index(doc_terms):
    index = {}
    for doc_id, terms in enumerate(doc_terms):
        ...   # ✍️ add doc_id to the postings list of each distinct term
    return index
t0 = time.time()
index = build_index(doc_terms)
print(f"indexed in {time.time() - t0:.1f} s: {len(index):,} terms")
print("leak:", len(index["leak"]), "reviews, starting", index["leak"][:8])

In [ ]:
# ✅ check against a brute-force scan of every review
for term in ["leak", "ice", "ukf8001"]:
    slow = [d for d, terms in enumerate(doc_terms) if term in terms]   # look through all 30,000 reviews
    assert index[term] == slow, f"postings for '{term}' differ from a full scan: duplicates, or missing reviews?"
print("✅ Why are the postings lists already sorted, although you never called sorted()?")

Now compare the two structures. The incidence matrix has a cell for every (term, review) pair. The inverted index
stores only the pairs that are present.

In [ ]:
cells = len(index) * len(doc_terms)                    # rows × columns of the full incidence matrix
postings = sum(len(p) for p in index.values())         # entries in all the postings lists = the 1s in the matrix
print(f"incidence matrix: {len(index):,} × {len(doc_terms):,} = {cells:,} cells")
print(f"inverted index:   {postings:,} postings, so {postings / cells:.2%} of the matrix is 1s")

> 📖 **Reading the output.** The matrix would have about 360 million cells, and fewer than 1 in 400 of them would be 1.
> A matrix that is almost all zeros is called **sparse**. The inverted index stores just the 1s, under a million entries,
> and that is why every search engine, from a library catalogue to Google, is built on one.

## Task 3 · AND: merging two postings lists (12 min)

To answer `ice AND leak`, find the doc IDs that are in **both** postings lists. Because both lists are sorted, you can walk
along them together, like merging two sorted piles of paper, and never go back:

```
i, j = 0, 0                                 # a pointer into each list
while both pointers are inside their lists:
    if p1[i] == p2[j]:  keep it, move both pointers on
    elif p1[i] < p2[j]: move i on           # p1[i] can't be in p2: everything after p2[j] is bigger
    else:               move j on
```

This takes at most `len(p1) + len(p2)` steps. Checking every pair of IDs would take `len(p1) × len(p2)`.

### ✍️ Exercise 3: `intersect(p1, p2)`

Write it with the two pointers above. Don't use `set`: the check compares your answer with one.

In [ ]:
def intersect(p1, p2):
    answer = []
    i, j = 0, 0
    while i < len(p1) and j < len(p2):
        ...   # ✍️ three cases: equal, p1[i] smaller, p2[j] smaller
    return answer
print(intersect([2, 4, 8, 16, 32, 64], [1, 2, 3, 5, 8, 13, 21, 34, 55]))

In [ ]:
# ✅ checks: a small case, edge cases, and real postings against Python's set intersection
assert intersect([2, 4, 8, 16, 32, 64], [1, 2, 3, 5, 8, 13, 21, 34, 55]) == [2, 8], "not quite on the small example"
assert intersect([], [1, 2]) == [] and intersect([5], [5]) == [5], "check the empty-list and one-item cases"
for a, b in [("ice", "leak"), ("water", "filter"), ("great", "not")]:
    assert intersect(index[a], index[b]) == sorted(set(index[a]) & set(index[b])), f"wrong for {a} AND {b}"
print("✅")
show(intersect(index["ice"], index["leak"]))

## Task 4 · OR and AND NOT (6 min)

`union` is the same walk, but it keeps **every** ID, and takes whatever is left at the end of either list. It is written
for you; read it.

In [ ]:
def union(p1, p2):
    answer = []
    i, j = 0, 0
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:                  # in both: keep it once
            answer.append(p1[i]); i += 1; j += 1
        elif p1[i] < p2[j]:                 # keep the smaller one and move past it
            answer.append(p1[i]); i += 1
        else:
            answer.append(p2[j]); j += 1
    return answer + p1[i:] + p2[j:]         # one list ran out: everything left in the other is in the answer

assert union([1, 3, 5], [2, 3, 9, 10]) == [1, 2, 3, 5, 9, 10]
print(len(index["fridg"]), "+", len(index["refriger"]), "→", len(union(index["fridg"], index["refriger"])), "reviews mention a fridge or a refrigerator")

### ✍️ Exercise 4: `and_not(p1, p2)`

Return the IDs in `p1` that are **not** in `p2`, for queries like `leak AND NOT coffee`. Use the same walk. Hint: when
`p1` has an ID that `p2` has skipped past, it is safe to keep; and don't forget what is left of `p1` at the end.

In [ ]:
def and_not(p1, p2):
    answer = []
    i, j = 0, 0
    while i < len(p1) and j < len(p2):
        ...   # ✍️ three cases again; only keep an ID from p1 when you know p2 doesn't have it
    return answer + ...   # ✍️ what is left of p1
print(and_not([1, 2, 3, 4, 5], [2, 4, 6]))

In [ ]:
# ✅ check
assert and_not([1, 2, 3, 4, 5], [2, 4, 6]) == [1, 3, 5], "not quite on the small example"
assert and_not([7, 8], []) == [7, 8], "when p2 is empty, keep all of p1"
assert and_not(index["leak"], index["coffe"]) == sorted(set(index["leak"]) - set(index["coffe"])), "wrong for leak AND NOT coffee"
print("✅")
show(and_not(index["leak"], index["coffe"]))

> 💬 Why does the engine support `leak AND NOT coffee`, but not plain `NOT coffee`? What would the answer to `NOT coffee` look like?

## Task 5 · Query planning: which list first? (6 min)

A query like `great water filter leak` is three ANDs. The answer is the same in any order, but the **work** is not.
After the first AND, every later step only has to check the **candidates** that survived so far. (Real engines don't
walk the whole of the longer list: they jump ahead in it with *skip pointers*, IIR section 2.3. So the work is driven by
the number of candidates.) `candidates` intersects the lists in the order given and adds up the candidates checked at each step.

In [ ]:
def candidates(postings_lists):
    # Intersect the lists in the order given. At each step, count the candidates we check against the next list,
    # and record how many survive.
    result, checked, sizes = postings_lists[0], 0, [len(postings_lists[0])]
    for p in postings_lists[1:]:
        checked += len(result)               # every surviving candidate is looked up in the next list
        result = sorted(set(result) & set(p))
        sizes.append(len(result))
    return checked, sizes

terms = normalise("great water filter leak")
print({t: len(index[t]) for t in terms})                                 # postings length of each term
for label, order in [("as typed", terms), ("shortest first", sorted(terms, key=lambda t: len(index[t])))]:
    checked, sizes = candidates([index[t] for t in order])
    print(f"{label:<15} {' → '.join(order):<34} candidates after each step {sizes}   checked {checked:,}")

> 📖 **Reading the output.** Typed order starts from the 6,531 *great* reviews and checks thousands of candidates
> before the short `leak` list cuts them down at the end. Starting from the 885 *leak* reviews keeps every
> intermediate result small, so it checks several times fewer candidates for the same answer. **Rule: process terms in order of increasing postings length.** The index already
> knows each length, the term's **document frequency**, so planning is free.

### ✍️ Exercise 5: `and_query(text)`

Write `and_query(text)`: normalise the query, look up each term's postings (`index.get(t, [])` gives `[]` for a term
that isn't in the index), sort them **shortest first**, then intersect them one by one. Stop early if the result is empty.

In [ ]:
def and_query(text):
    lists = ...    # ✍️ the postings list of each term in normalise(text), sorted by length
    result = lists[0]
    for p in lists[1:]:
        ...        # ✍️ stop if result is empty, otherwise intersect
    return result
show(and_query("ice maker leaking"))

In [ ]:
# ✅ check against a brute-force scan
def slow_and(text):
    want = set(normalise(text))
    return [d for d, terms in enumerate(doc_terms) if want <= set(terms)]   # <= : every query term is in the review
for q in ["ice maker leaking", "Great water filter, leaks", "warranty refund crack", "zzzz filter"]:
    assert and_query(q) == slow_and(q), f"wrong for '{q}'"
t0 = time.perf_counter(); and_query("great water filter leak"); fast = time.perf_counter() - t0
t0 = time.perf_counter(); slow_and("great water filter leak"); slow = time.perf_counter() - t0
print(f"✅ index: {fast * 1000:.2f} ms   full scan: {slow * 1000:.0f} ms")

## Task 6 · The category manager's questions (10 min)

Use `and_query`, `union`, `and_not` and `index[...]` to answer each question. Remember that index terms are **stems**:
use `normalise("word")` to see a word's stem, or go through `and_query`, which normalises for you.
Each check compares your result with a brute-force answer, so any correct combination passes.

**Q1.** Which reviews mention an **ice maker** and a **leak**, but not **water filters**?
(`ice maker leak`, AND NOT anything containing `filter`)

In [ ]:
q1 = ...   # ✍️
assert q1 == [d for d in slow_and("ice maker leak") if "filter" not in doc_terms[d]], "not quite"
print("✅"); show(q1, n=5)

**Q2.** Water-filter reviews that complain about **taste or smell**: `water AND filter AND (taste OR smell)`.

In [ ]:
q2 = ...   # ✍️ hint: intersect(<water filter>, union(<taste>, <smell>))
assert q2 == [d for d in slow_and("water filter") if {"tast", "smell"} & set(doc_terms[d])], "not quite"
print("✅"); show(q2)
print(f"average rating {reviews.loc[q2, 'rating'].mean():.2f}, against {reviews['rating'].mean():.2f} overall")

> 📖 **Reading the output.** Over 900 reviews, and their average rating is barely below the overall average. Read the
> first two: *"the water tastes great"*. The query finds reviews that **mention** taste, not reviews that **complain** about it.

**Q3.** Run these four queries and look at the number of results. What would you do with each list, as a manager?

In [ ]:
for q in ["filter", "filter leak", "filter leak crack", "filter leak crack warranty"]:
    print(f"{q:<28} {len(and_query(q)):>5} results")

> 📖 **Reading the output.** One word gives thousands of reviews, and four words give none. This is the classic problem
> with Boolean search, called **feast or famine**: the user gets too many results or none, and the results come in
> no particular order. Which of the 355 *filter leak* reviews should the manager read first? Boolean search can't say.
> **Ranking** by how well each review matches is Workshop 5 (TF-IDF).

**Q4.** Synonyms. Customers say *fridge* and *refrigerator* (and sometimes *freezer*). How many reviews mention leaking
with **each** word, and with **either**?

In [ ]:
fridge_leak = and_query("fridge leak")
fridgerator_leak = and_query("refrigerator leak")
either = intersect(union(index[stem("fridge")], index[stem("refrigerator")]), index["leak"])
print(f"fridge AND leak: {len(fridge_leak)}   refrigerator AND leak: {len(fridgerator_leak)}   either, AND leak: {len(either)}")

> 💬 **Discuss:** Stemming can't merge *fridge* and *refrigerator*: they share no ending to strip. To find both, the
> engine needs a list of synonyms, or a way to learn that the two words mean the same thing. That is **word vectors**,
> Workshop 6.

## Task 7 · Stretch: phrase queries

`ice AND maker` also matches *"the ice was fine but the coffee maker broke"*. To search for the **phrase** *"ice maker"*,
the index must also store **where** each term occurs. A **positional index** maps each term to `{doc ID: [positions]}`.

In [ ]:
positional = {}
for doc_id, terms in enumerate(doc_terms):
    for pos, t in enumerate(terms):                                   # pos = 0, 1, 2, ... the word's place in the review
        positional.setdefault(t, {}).setdefault(doc_id, []).append(pos)
first = next(iter(positional["ice"]))                                 # the first review that contains "ice"
print(first, positional["ice"][first], doc_terms[first][:12])        # its ID, where "ice" occurs, and its first 12 terms

### ✍️ Exercise 7: `phrase2(w1, w2)`

Return the sorted doc IDs where `w2` comes **immediately after** `w1`: some position `p` of `w1` has `p + 1` among the positions of `w2`.
Start from the docs that contain both words (`intersect` on the keys), then check positions.

In [ ]:
def phrase2(w1, w2):
    t1, t2 = stem(w1), stem(w2)
    both = ...   # ✍️ doc IDs that contain both terms
    answer = []
    for d in both:
        ...      # ✍️ keep d if some position of t1, plus 1, is a position of t2
    return answer
ice_maker = phrase2("ice", "maker")
assert ice_maker == [d for d, ts in enumerate(doc_terms) if any(a == "ice" and b == "maker" for a, b in zip(ts, ts[1:]))]
print("✅")
for w1, w2 in [("ice", "maker"), ("water", "filter"), ("not", "leak")]:
    print(f"{w1} AND {w2}: {len(and_query(w1 + ' ' + w2)):>5}   phrase \"{w1} {w2}\": {len(phrase2(w1, w2)):>5}")

> 📖 **Reading the output.** For *ice maker* the phrase changes little, because the two words nearly always come together.
> For *water filter* it cuts 2,409 reviews to 622: most reviews that use both words aren't talking about a water filter
> as a thing. And the phrase *"not leak"* finds the 48 reviews that say the product does **not** leak (more if you add
> *doesn't*, *no* and *never*). A query can use word order as well as words.

## Before the share-out

Bring answers to these:

1. **Feast or famine.** Your best query for *"filters that leak"*: how many results, and how would you choose which to read first?
2. **No leaks.** Read 10 results of `and_query("filter leak")`. How many are complaints? Is that a failure of your index, or of Boolean search?
3. **Normalisation.** Your index stems everything. Name one query where that helps the category team, and one where it hurts.
4. **Scale.** Your index answered in about a millisecond, against about 20 ms for a full scan. What happens to each as the collection grows from 30,000 reviews to 300 million?